## 1. Installing Required Packages

In [ ]:
# =============================================================================
# Installing required packages
# =============================================================================
!pip install -U scikit-learn scikeras --quiet
!pip install statsmodels optuna tabulate xgboost scikeras --quiet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 84.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 17.6 MB/s eta 0:00:00


## 2. Imports & Global Settings

In [ ]:
# Standard libraries
import sys
import os
import warnings
import random
warnings.filterwarnings("ignore")

# Data manipulation
import numpy as np   # Numerical operations
import pandas as pd  # Data manipulation and analysis

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Preprocessing & model selection
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (TimeSeriesSplit, GridSearchCV,
                                     RandomizedSearchCV)
from sklearn.metrics import (mean_squared_error, mean_absolute_error, r2_score)
from sklearn.inspection import permutation_importance
from sklearn.neural_network import MLPRegressor
from scipy.stats import randint, uniform

# XGBoost
import xgboost as xgb

# TensorFlow / Keras (LSTM)
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Conv1D, Flatten
from tensorflow.keras.optimizers import Adam

# SciKeras wrapper (scikit-learn API for Keras)
from scikeras.wrappers import KerasRegressor

# Optuna (Bayesian hyperparameter optimisation)
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)  # suppress verbose trial logs

# ── Global random seed (set once; reused throughout) ──────────────────────────
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)
print(f"Global random seed set to {RANDOM_SEED}")


Global random seed set to 42


## 3. Loading the Dataset

In [ ]:
# =============================================================================
# LOADING THE DATASET
# Dataset sources:
#   https://drive.google.com/file/d/1Z_KsoIumw-fvivVombIoWuRo0LOe2nCb/view?usp=sharing
#   https://drive.google.com/file/d/1aD1PXfwEEZ_F2lQgxuPfj-TVbxQ6NajK/view?usp=sharing
# =============================================================================

file_id      = "1c5KmcFD1TOsqASqzy0KiYo-7IzgBFuhk"   # Variable Set 2 dataset
download_url = f"https://drive.google.com/uc?id={file_id}"

df = pd.read_csv(download_url)

# Create a proper datetime index from year / month / day columns
df['Date'] = pd.to_datetime({
    'year':  df['YEAR'],
    'month': df['MO'],
    'day':   df['DY']
})
df = df.set_index('Date')

# Drop the raw date columns (now encoded in the index)
df = df.drop(columns=["YEAR", "MO", "DY"])

print("Dataset loaded successfully.")
print(f"Shape: {df.shape}")
print("\nFirst 5 rows:")
display(df.head())
print("\nColumns:", df.columns.tolist())


Dataset loaded successfully.
Shape: (4015, 14)

First 5 rows:


,WS10M_lag1,RH,MIN_TEMP,PREC,WD_sin,SURF_PRESSURE_DIFF,AVG_TEMP,WD_cos,RH_lag1,MONOSOON_SEASON_Southwest_Monsoon,MAX_TEMP,SL_PRESSURE_lag1,CLOUD_COVER,WS10M
Date,,,,,,,,,,,,,,
2013-01-03,4.25,86.26,23.42,13.42,0.439939,0.03,26.4,0.898028,86.19,0,28.38,1010.4,8.0,4.75
2013-01-04,4.75,86.31,22.93,8.79,0.424199,-0.03,25.1,0.905569,86.26,0,27.58,1011.0,8.0,5.74
2013-01-05,5.74,86.88,22.17,2.60,0.563526,-0.15,26.9,0.826098,86.31,0,26.16,1010.1,8.0,5.79
2013-01-06,5.79,88.09,23.47,1.65,0.460200,-0.03,27.3,0.887815,86.88,0,27.75,1007.5,7.0,4.52
2013-01-07,4.52,93.04,24.04,27.41,0.368125,-0.01,25.9,0.929776,88.09,0,26.70,1008.2,8.0,4.66



Columns: ['WS10M_lag1', 'RH', 'MIN_TEMP', 'PREC', 'WD_sin', 'SURF_PRESSURE_DIFF', 'AVG_TEMP', 'WD_cos', 'RH_lag1', 'MONOSOON_SEASON_Southwest_Monsoon', 'MAX_TEMP', 'SL_PRESSURE_lag1', 'CLOUD_COVER', 'WS10M']


## 4. Primary Train-Test Split (80/20, Time-Based)

In [ ]:
# =============================================================================
# Train-test split — first 80% for training, last 20% for testing.
# Temporal ordering is strictly preserved; no shuffling is applied.
# =============================================================================

X = df.drop(columns=['WS10M'])  # Feature matrix
y = df['WS10M']                 # Target: wind speed (m/s)

split_index = int(len(X) * 0.8)  # 80% training, 20% testing

X_train = X.iloc[:split_index]
y_train = y.iloc[:split_index]
X_test  = X.iloc[split_index:]
y_test  = y.iloc[split_index:]

print(f"Total samples : {len(X)}")
print(f"Training set  : {X_train.shape}  (first {split_index} samples)")
print(f"Test set      : {X_test.shape}  (last {len(X) - split_index} samples)")
print(f"\nTrain period: {X_train.index[0].date()}  →  {X_train.index[-1].date()}")
print(f"Test period : {X_test.index[0].date()}  →  {X_test.index[-1].date()}")


Total samples : 4015
Training set  : (3212, 13)  (first 3212 samples)
Test set      : (803, 13)  (last 803 samples)

Train period: 2013-01-03  →  2021-10-19
Test period : 2021-10-20  →  2023-12-31


## 5. Shared Helper Functions

In [ ]:
# =============================================================================
# Helper functions used by all model sections below.
# Defined once here to avoid repetition.
# =============================================================================

def evaluate(y_true, y_pred, label=""):
    """Print and return evaluation metrics for the residual correction model."""
    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae  = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    r2   = r2_score(y_true, y_pred)
    print(f"{label} MSE:  {mse:.4f}")
    print(f"{label} RMSE: {rmse:.4f}")
    print(f"{label} MAE:  {mae:.4f}")
    print(f"{label} MAPE: {mape:.2f}%")
    print(f"{label} R²:   {r2:.4f}")
    return mse, rmse, mae, mape, r2


def evaluate_performance(y_true, y_pred, set_name="Dataset"):
    """Evaluate and return metrics as a dictionary for the hybrid model."""
    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae  = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    r2   = r2_score(y_true, y_pred)
    print(f"\n--- {set_name} Metrics ---")
    print(f"RMSE: {rmse:.4f}")
    print(f"MSE:  {mse:.4f}")
    print(f"MAE:  {mae:.4f}")
    print(f"MAPE: {mape:.4f}%")
    print(f"R²:   {r2:.4f}")
    return {'RMSE': rmse, 'MSE': mse, 'MAE': mae, 'MAPE': mape, 'R²': r2}


## 6. XGBoost Primary Model (Optuna-Optimised Parameters)

In [ ]:
# =============================================================================
# XGBoost Primary Model
# Best parameters sourced from prior Optuna optimisation run.
# The model is fitted on X_train only; predictions over the full dataset are
# stored so residuals can be computed for the LSTM correction layer.
# =============================================================================

# Best Parameters: {'n_estimators': 300, 'max_depth': 6,
#   'learning_rate': 0.02017215364230889, 'min_child_weight': 5,
#   'gamma': 0.019237778052053844, 'subsample': 0.618408743715084}
Best_Parameters = {
    'n_estimators':     300,
    'max_depth':        6,
    'learning_rate':    0.02017215364230889,
    'gamma':            0.019237778052053844,
    'subsample':        0.618408743715084,
    'min_child_weight': 5
}

# Initialise XGBRegressor with optimal parameters
best_xgb = xgb.XGBRegressor(**Best_Parameters, random_state=RANDOM_SEED)

# Fit on training data only (test set remains unseen until evaluation)
best_xgb.fit(X_train, y_train)
print(f"XGBoost fitted. Parameters: {Best_Parameters}")

# ── Train / Test predictions ──────────────────────────────────────────────────
y_train_pred_xgb = best_xgb.predict(X_train)
y_test_pred_xgb  = best_xgb.predict(X_test)

# ── Evaluation metrics ────────────────────────────────────────────────────────
xgb_train_mse  = mean_squared_error(y_train, y_train_pred_xgb)
xgb_test_mse   = mean_squared_error(y_test,  y_test_pred_xgb)
xgb_train_mae  = mean_absolute_error(y_train, y_train_pred_xgb)
xgb_test_mae   = mean_absolute_error(y_test,  y_test_pred_xgb)
xgb_train_rmse = np.sqrt(xgb_train_mse)
xgb_test_rmse  = np.sqrt(xgb_test_mse)
xgb_train_mape = np.mean(np.abs((y_train - y_train_pred_xgb) / y_train)) * 100
xgb_test_mape  = np.mean(np.abs((y_test  - y_test_pred_xgb)  / y_test))  * 100
xgb_train_r2   = r2_score(y_train, y_train_pred_xgb)
xgb_test_r2    = r2_score(y_test,  y_test_pred_xgb)

print(f"\nTrain MSE:  {xgb_train_mse:.4f}  | Test MSE:  {xgb_test_mse:.4f}")
print(f"Train MAE:  {xgb_train_mae:.4f}  | Test MAE:  {xgb_test_mae:.4f}")
print(f"Train RMSE: {xgb_train_rmse:.4f} | Test RMSE: {xgb_test_rmse:.4f}")
print(f"Train MAPE: {xgb_train_mape:.4f}%| Test MAPE: {xgb_test_mape:.4f}%")
print(f"Train R²:   {xgb_train_r2:.4f}  | Test R²:   {xgb_test_r2:.4f}")

# ── Summary table (Plotly) ────────────────────────────────────────────────────
summary_df = pd.DataFrame({
    'Parameter/Metric': list(Best_Parameters.keys()) +
                        ['Train MSE', 'Test MSE', 'Train MAE', 'Test MAE',
                         'Train RMSE', 'Test RMSE', 'Train MAPE (%)',
                         'Test MAPE (%)', 'Train R²', 'Test R²'],
    'Value': list(Best_Parameters.values()) +
             [xgb_train_mse, xgb_test_mse, xgb_train_mae, xgb_test_mae,
              xgb_train_rmse, xgb_test_rmse, xgb_train_mape, xgb_test_mape,
              xgb_train_r2, xgb_test_r2]
})
summary_df['Value'] = summary_df['Value'].apply(lambda x: f"{x:.6f}")

fig_table = go.Figure(go.Table(
    header=dict(
        values=['<b>Parameter / Metric</b>', '<b>Value</b>'],
        fill_color='steelblue',
        font=dict(color='white', size=13),
        align='left'
    ),
    cells=dict(
        values=[summary_df['Parameter/Metric'], summary_df['Value']],
        fill_color=[['lightcyan' if i % 2 == 0 else 'white'
                     for i in range(len(summary_df))]],
        align='left'
    )
))
fig_table.update_layout(
    title="XGBoost Optimal Parameters & Model Performance Metrics",
    template="plotly_white", height=600
)
fig_table.show()


XGBoost fitted. Parameters: {'n_estimators': 300, 'max_depth': 6, 'learning_rate': 0.02017215364230889, 'gamma': 0.019237778052053844, 'subsample': 0.618408743715084, 'min_child_weight': 5}

Train MSE:  0.2103  | Test MSE:  0.5259
Train MAE:  0.3590  | Test MAE:  0.5495
Train RMSE: 0.4586 | Test RMSE: 0.7252
Train MAPE: 9.9187%| Test MAPE: 14.9222%
Train R²:   0.9165  | Test R²:   0.7819


## 7. Shared Cross-Conformal Prediction-Interval Engine

In [ ]:
# =============================================================================
# SHARED CROSS-CONFORMAL ENGINE  (run this cell ONCE, before any model cell)
# -----------------------------------------------------------------------------
# FIXES vs. the original Section 14 code:
#
#   BUG 1 (fatal, stopped everything downstream of it):
#     `y_actual_train_full` / `xgb_pred_train_full` were sliced POSITIONALLY
#     from df_backup (length = split_index_res, i.e. the FULL, un-filtered
#     80% training block), while `X_train_res` / `y_train_res` had already
#     been filtered to drop NaN rows (the initial OOF-uncovered block, plus
#     the first 1-2 rows with no lag feature). Those two arrays are therefore
#     SHORTER, so `assert len(y_actual_train_full) == len(X_train_res)`
#     raised an AssertionError immediately -- before even the standalone XGB
#     calibration cell (which doesn't depend on the buggy variables) could
#     run. Even without the assert, feeding same-length-required arrays that
#     are actually misaligned would have silently paired each validation
#     fold's predictions with the WRONG actual/base-XGB rows.
#     FIX: `get_hybrid_train_calibration_arrays()` below re-derives the
#     residual-model features straight from df_backup and re-selects the
#     matching actual/base-XGB rows using `df_backup.loc[X_train_res.index]`
#     -- i.e. by DATE INDEX, not by position -- so alignment is guaranteed
#     regardless of how many rows were dropped upstream.
#
#   BUG 2 (fragility): the six calibration cells were one monolithic block
#     sharing global variable names with the model-training sections earlier
#     in the notebook, so one failing model (e.g. FNN not yet trained) broke
#     the whole six-way pd.concat and the comparison table.
#     FIX: every model below is a fully independent, try/except-wrapped cell
#     that only reads df_backup + its own tuned hyperparameters. Run them in
#     any order; any one can fail without affecting the others.
# =============================================================================

import numpy as np
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Conv1D, Flatten
from tensorflow.keras.optimizers import Adam
from sklearn.neural_network import MLPRegressor

CONFIDENCE_LEVELS = {'90%': 0.10, '95%': 0.05}


def get_hybrid_train_calibration_arrays(df_backup, split_index_res,
                                         feature_cols=('Residual_lag1', 'Residual_lag2')):
    """Self-contained: rebuilds the residual-model calibration features
    directly from df_backup, then aligns the matching actual WS10M and fixed
    XGBoost predictions by DATE INDEX (not position) -- this is the fix for
    BUG 1 above."""
    feats = list(feature_cols)
    X_res_full = df_backup[feats].iloc[:split_index_res]
    y_res_full = df_backup['Residual'].iloc[:split_index_res]
    valid_mask = X_res_full.notna().all(axis=1) & y_res_full.notna()

    X_train_res = X_res_full.loc[valid_mask]
    y_train_res = y_res_full.loc[valid_mask]
    y_actual_aligned = df_backup.loc[X_train_res.index, 'WS10M']
    xgb_pred_aligned  = df_backup.loc[X_train_res.index, 'Pred_XGB_Optuna']

    assert len(X_train_res) == len(y_train_res) == len(y_actual_aligned) == len(xgb_pred_aligned), \
        "Index misalignment while rebuilding hybrid calibration arrays -- please check df_backup."
    return X_train_res, y_train_res, y_actual_aligned, xgb_pred_aligned


def get_hybrid_test_point_forecast(df_backup, split_index_res, hybrid_col):
    """Pulls the already-computed test-set hybrid forecast (fixed XGB + tuned
    residual model) straight from df_backup, drops any NaN rows, and returns
    it aligned with the matching actual WS10M values."""
    y_pred_test = df_backup[hybrid_col].iloc[split_index_res:].dropna()
    y_actual_test = df_backup.loc[y_pred_test.index, 'WS10M']
    return y_actual_test, y_pred_test


def cross_conformal_pooled_errors(X_arr, y_fold_target_arr, y_actual_arr,
                                   base_pred_arr, fold_predict_fn,
                                   n_splits=5, variant_label="",
                                   is_residual_model=True):
    """Pools out-of-fold |actual - forecast| across a 5-fold TimeSeriesSplit.
    Unchanged core logic -- correctness now only requires its four array
    arguments to already be the same length and positionally aligned, which
    the two helpers above guarantee."""
    tscv = TimeSeriesSplit(n_splits=n_splits)
    pooled_abs_residuals = []
    fold_records = []

    for fold_i, (train_idx, val_idx) in enumerate(tscv.split(X_arr), start=1):
        X_t, X_v = X_arr[train_idx], X_arr[val_idx]
        y_t      = y_fold_target_arr[train_idx]

        pred_val = fold_predict_fn(X_t, y_t, X_v)

        if is_residual_model:
            forecast_val = base_pred_arr[val_idx] + pred_val
        else:
            forecast_val = pred_val

        actual_val = y_actual_arr[val_idx]
        abs_res = np.abs(actual_val - forecast_val)
        pooled_abs_residuals.extend(abs_res.tolist())

        fold_records.append({
            'Variant': variant_label, 'Fold': fold_i, 'Val_size': len(val_idx),
            'Mean_abs_forecast_error': abs_res.mean(),
            'Median_abs_forecast_error': np.median(abs_res)
        })
        print(f"[{variant_label}] Fold {fold_i}/{n_splits} "
              f"(val size={len(val_idx):4d}) - mean |forecast error| = {abs_res.mean():.4f}")

    return np.array(pooled_abs_residuals), pd.DataFrame(fold_records)


def conformal_quantile(abs_residuals, alpha=0.05):
    """Finite-sample-corrected split-conformal quantile. Guarantees marginal
    coverage of at least (1 - alpha) under exchangeability."""
    n = len(abs_residuals)
    q_level = min(np.ceil((n + 1) * (1 - alpha)) / n, 1.0)
    return np.quantile(abs_residuals, q_level, method='higher')


def evaluate_prediction_interval(y_true, y_pred, q):
    """Empirical coverage (%) and average interval width on the test set."""
    lower = y_pred - q
    upper = y_pred + q
    covered = (y_true >= lower) & (y_true <= upper)
    return lower, upper, covered.mean() * 100, (upper - lower).mean()


def build_pi_summary(variant_name, pooled_abs_residuals, y_actual_test, y_pred_test):
    """Builds the 90%/95% prediction-interval rows for one model.
    Returns (summary_df, bounds_dict) where bounds_dict['95%'] = (lower, upper)."""
    rows, bounds = [], {}
    for level_name, alpha in CONFIDENCE_LEVELS.items():
        q = conformal_quantile(pooled_abs_residuals, alpha=alpha)
        lower, upper, coverage, width = evaluate_prediction_interval(
            y_actual_test.values, y_pred_test.values, q)
        bounds[level_name] = (lower, upper)
        rows.append({
            'Model': variant_name, 'Confidence Level': level_name,
            'Calibration n': len(pooled_abs_residuals),
            'Margin (q, m/s)': round(q, 4),
            'Empirical Coverage (%)': round(coverage, 2),
            'Mean Interval Width (m/s)': round(width, 4)
        })
    return pd.DataFrame(rows), bounds


# ── Fold-level model factories (rebuild each already-tuned architecture per fold) ──

def build_fold_cnn(n_features, filters, dropout_rate, learning_rate):
    mdl = Sequential([
        Conv1D(filters=filters, kernel_size=1, activation='relu', input_shape=(1, n_features)),
        Flatten(), Dropout(dropout_rate), Dense(50, activation='relu'), Dense(1)
    ])
    mdl.compile(loss='mse', optimizer=Adam(learning_rate=learning_rate))
    return mdl


def build_fold_lstm_grid(n_features, units, dropout_rate, optimizer):
    mdl = Sequential([
        LSTM(units, activation='tanh', return_sequences=False, input_shape=(1, n_features)),
        Dropout(dropout_rate), Dense(1)
    ])
    mdl.compile(optimizer=optimizer, loss='mse')
    return mdl


def build_fold_lstm_adam(n_features, units, dropout_rate, learning_rate):
    mdl = Sequential([
        LSTM(units=units, input_shape=(1, n_features)),
        Dropout(dropout_rate), Dense(1)
    ])
    mdl.compile(loss='mse', optimizer=Adam(learning_rate=learning_rate))
    return mdl


def make_keras_fold_fn(model_builder, build_kwargs, batch_size, epochs, random_seed=42):
    """Wraps a Keras model builder into a fold_predict_fn (CNN / LSTM variants)."""
    def _fn(X_t, y_t, X_v):
        fold_scaler = StandardScaler()
        X_t_sc = fold_scaler.fit_transform(X_t)
        X_v_sc = fold_scaler.transform(X_v)
        X_t_3d = np.expand_dims(X_t_sc, axis=1)
        X_v_3d = np.expand_dims(X_v_sc, axis=1)
        n_features = X_t_3d.shape[2]
        tf.random.set_seed(random_seed)
        fold_model = model_builder(n_features, **build_kwargs)
        fold_model.fit(X_t_3d, y_t, epochs=epochs, batch_size=batch_size, verbose=0, shuffle=False)
        return fold_model.predict(X_v_3d, verbose=0).flatten()
    return _fn


def make_mlp_fold_fn(mlp_kwargs, random_seed=42):
    """Wraps MLPRegressor into a fold_predict_fn (FNN variant)."""
    def _fn(X_t, y_t, X_v):
        fold_scaler = StandardScaler()
        X_t_sc = fold_scaler.fit_transform(X_t)
        X_v_sc = fold_scaler.transform(X_v)
        fold_model = MLPRegressor(random_state=random_seed, **mlp_kwargs)
        fold_model.fit(X_t_sc, y_t)
        return fold_model.predict(X_v_sc)
    return _fn


def make_xgb_standalone_fold_fn(xgb_params, random_seed=42):
    """Wraps XGBRegressor into a fold_predict_fn (standalone XGB Optuna variant)."""
    def _fn(X_t, y_t, X_v):
        fold_model = xgb.XGBRegressor(**xgb_params, random_state=random_seed)
        fold_model.fit(X_t, y_t)
        return fold_model.predict(X_v)
    return _fn


print("Shared cross-conformal engine (fixed, index-aligned) loaded.")


Shared cross-conformal engine (fixed, index-aligned) loaded.


## 8. Prediction Interval — XGBoost (Optuna, standalone)

Addresses reviewer comment: *"The authors should obtain prediction intervals for the best-performing model to quantify the associated forecast uncertainty."* Cross-conformal calibration on the standalone XGBoost model.

In [ ]:
# =============================================================================
# MODEL 1 / 6 — Prediction Interval: XGB Optuna (standalone benchmark)
# -----------------------------------------------------------------------------
# Independent cell. Requires only objects from Section 6 (XGB Primary Model):
#   X_train, y_train, X_test, y_test, Best_Parameters, y_test_pred_xgb,
#   RANDOM_SEED
# Run the ENGINE cell above once first. This cell does not read or write any
# variable used by the other five model cells below.
# =============================================================================
try:
    xgb_standalone_fold_fn = make_xgb_standalone_fold_fn(Best_Parameters, random_seed=RANDOM_SEED)

    pooled_residuals_xgb, fold_summary_xgb = cross_conformal_pooled_errors(
        X_arr              = X_train.values,
        y_fold_target_arr  = y_train.values,
        y_actual_arr       = y_train.values,
        base_pred_arr      = np.zeros(len(y_train)),   # unused -- no residual stage
        fold_predict_fn    = xgb_standalone_fold_fn,
        n_splits           = 5,
        variant_label       = "XGB (Optuna, standalone)",
        is_residual_model  = False
    )
    print(f"\nPooled calibration residuals (XGB standalone): n = {len(pooled_residuals_xgb)}")

    y_pred_test_xgb_series = pd.Series(y_test_pred_xgb, index=y_test.index)

    pi_summary_xgb, pi_bounds_xgb = build_pi_summary(
        "XGB (Optuna, standalone)", pooled_residuals_xgb, y_test, y_pred_test_xgb_series)

    print("\n=== Prediction Interval — XGB (Optuna, standalone) ===")
    print(pi_summary_xgb.to_string(index=False))
    pi_summary_xgb.to_csv('pi_summary_xgb_standalone.csv', index=False)
    print("Saved to 'pi_summary_xgb_standalone.csv'")

except Exception as e:
    print(f"\n[MODEL 1 -- XGB standalone] PI cell FAILED: {type(e).__name__}: {e}")
    print("(The other five model cells are unaffected and can still be run.)")
    pi_summary_xgb, pi_bounds_xgb = None, None


[XGB (Optuna, standalone)] Fold 1/5 (val size= 535) - mean |forecast error| = 0.5622
[XGB (Optuna, standalone)] Fold 2/5 (val size= 535) - mean |forecast error| = 0.5362
[XGB (Optuna, standalone)] Fold 3/5 (val size= 535) - mean |forecast error| = 0.5600
[XGB (Optuna, standalone)] Fold 4/5 (val size= 535) - mean |forecast error| = 0.5324
[XGB (Optuna, standalone)] Fold 5/5 (val size= 535) - mean |forecast error| = 0.5797

Pooled calibration residuals (XGB standalone): n = 2675

=== Prediction Interval — XGB (Optuna, standalone) ===
                   Model Confidence Level  Calibration n  Margin (q, m/s)  Empirical Coverage (%)  Mean Interval Width (m/s)
XGB (Optuna, standalone)              90%           2675           1.1599                   88.42                     2.3197
XGB (Optuna, standalone)              95%           2675           1.4591                   94.40                     2.9182
Saved to 'pi_summary_xgb_standalone.csv'
